In [ ]:
# 0/5 Diagnostic start marker (for headless debugging)print("KAGGLE_CELL1_MARKER", flush=True)

In [ ]:
# 1/5 Install Ollama, cloudflared, and GPU deps!apt-get update -qq && apt-get install -y -qq pciutils zstd >/dev/null 2>&1# Check GPU is visible (no driver = CPU-only = too slow for 14B)!nvidia-smi# Install Ollama (no systemd on Kaggle, so we start `serve` manually in cell 2)!curl -fsSL https://ollama.com/install.sh | sh# cloudflared quick tunnel binary!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared!chmod +x /usr/local/bin/cloudflared!echo --- && ollama --version && cloudflared --version

In [ ]:
# 2/5 Start the Ollama server in the backgroundimport os, subprocess, threading, socket, timedef wait_port(port=11434, timeout=120):    deadline = time.time() + timeout    while time.time() < deadline:        try:            with socket.create_connection(("127.0.0.1", port), timeout=2):                return True        except OSError:            time.sleep(1)    return Falsedef pipe_lines(proc, tag):    for line in proc.stdout:        print(f"[{tag}] {line}", end="")env = os.environ.copy()env.update({    "OLLAMA_HOST": "127.0.0.1:11434",    "OLLAMA_ORIGINS": "*",          # CORS for remote callers    "OLLAMA_NUM_CTX": "32768",      # tool-call friendly context    "OLLAMA_KEEP_ALIVE": "-1",      # keep model resident while session lives})proc = subprocess.Popen(    ["/usr/local/bin/ollama", "serve"],    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env,)threading.Thread(target=pipe_lines, args=(proc, "OLLAMA"), daemon=True).start()assert wait_port(), "Ollama did not start on 127.0.0.1:11434"print("\nOllama is serving on 127.0.0.1:11434")